# 01 · Data Cleaning & Feature Engineering
**Project:** Hospital Appointment & No-Show Analysis

**Goal of this notebook:** turn the raw Kaggle file into a clean, analysis-ready table.

Steps
1. Load and inspect the raw data
2. Check data quality (missing values, duplicates, impossible values)
3. Clean (rename, fix types, recode the confusing `No-show` column, drop invalid rows)
4. Engineer features (wait time, age group, weekday, patient history ...)
5. Validate and save to `data/processed/appointments_clean.csv`

> All logic lives in `src/data_prep.py` so the notebook, the models and the dashboard use exactly the same code.

In [1]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import data_prep as dp

pd.set_option("display.max_columns", 50)
RAW = "../data/raw/KaggleV2-May-2016.csv"
OUT = "../data/processed/appointments_clean.csv"


## 1 · Load and inspect the raw data

In [2]:
raw = dp.load_raw(RAW)
print("Shape:", raw.shape)
raw.head()

Shape: (110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  str    
 3   ScheduledDay    110527 non-null  str    
 4   AppointmentDay  110527 non-null  str    
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  str    
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  str    
dtypes: float64(1), int64(8), str(5)
memory usage: 17.6 MB


## 2 · Data-quality checks

In [4]:
checks = pd.Series({
    "Missing values (total)": int(raw.isna().sum().sum()),
    "Fully duplicated rows": int(raw.duplicated().sum()),
    "Duplicated AppointmentID": int(raw["AppointmentID"].duplicated().sum()),
    "Unique patients": int(raw["PatientId"].nunique()),
    "Age < 0": int((raw["Age"] < 0).sum()),
    "Age == 0 (infants, valid)": int((raw["Age"] == 0).sum()),
    "Age > 100": int((raw["Age"] > 100).sum()),
})
checks.to_frame("count")

,count
Missing values (total),0
Fully duplicated rows,0
Duplicated AppointmentID,0
Unique patients,62299
Age < 0,1
"Age == 0 (infants, valid)",3539
Age > 100,7


In [5]:
# Appointment date is stored as midnight, so compare dates only
sched = pd.to_datetime(raw["ScheduledDay"]).dt.tz_localize(None).dt.normalize()
appt = pd.to_datetime(raw["AppointmentDay"]).dt.tz_localize(None).dt.normalize()
print("Appointments booked AFTER the appointment date:", int((appt < sched).sum()))
print("Booking dates   :", sched.min().date(), "->", sched.max().date())
print("Appointment dates:", appt.min().date(), "->", appt.max().date())
print("\nHandcap values:\n", raw["Handcap"].value_counts().to_string())
print("\nNo-show values (careful: 'Yes' = patient did NOT come):\n", raw["No-show"].value_counts().to_string())

Appointments booked AFTER the appointment date: 5
Booking dates   : 2015-11-10 -> 2016-06-08
Appointment dates: 2016-04-29 -> 2016-06-08

Handcap values:
 Handcap
0    108286
1      2042
2       183
3        13
4         3

No-show values (careful: 'Yes' = patient did NOT come):
 No-show
No     88208
Yes    22319


**What we found**

| Issue | Rows | Decision |
|---|---|---|
| No missing values, no duplicates | 0 | nothing to do |
| `Age = -1` | 1 | drop (impossible) |
| Appointment date earlier than booking date | 5 | drop (impossible) |
| `Age = 0` | 3,539 | **keep** - these are real infants (many original analyses wrongly drop them) |
| `Handcap` has values 0-4 | - | treat as a yes/no flag, keep the level separately |
| Typos: `Hipertension`, `Handcap` | - | rename |
| `No-show = Yes` means the patient missed | - | recode to `missed` (1 = missed) |

## 3 · Clean

In [6]:
df = dp.clean(raw)
df.head()

Rows in: 110,527 | invalid age (<0): 1 | appointment before booking: 5 | rows out: 110,521


,patient_id,appointment_id,gender,scheduled_dt,appointment_date,age,neighbourhood,scholarship,hypertension,diabetes,alcoholism,handicap,sms_received,scheduled_date,missed,wait_days,handicap_level
0,29872499824296,5642903,F,2016-04-29 18:38:08,2016-04-29,62,JARDIM DA PENHA,0,1,0,0,0,0,2016-04-29,0,0,0
1,558997776694438,5642503,M,2016-04-29 16:08:27,2016-04-29,56,JARDIM DA PENHA,0,0,0,0,0,0,2016-04-29,0,0,0
2,4262962299951,5642549,F,2016-04-29 16:19:04,2016-04-29,62,MATA DA PRAIA,0,0,0,0,0,0,2016-04-29,0,0,0
3,867951213174,5642828,F,2016-04-29 17:29:31,2016-04-29,8,PONTAL DE CAMBURI,0,0,0,0,0,0,2016-04-29,0,0,0
4,8841186448183,5642494,F,2016-04-29 16:07:23,2016-04-29,56,JARDIM DA PENHA,0,1,1,0,0,0,2016-04-29,0,0,0


## 4 · Feature engineering

In [7]:
df = dp.add_features(df)
new_cols = ["wait_days", "wait_group", "same_day", "age_group", "appointment_weekday", "scheduled_hour",
            "num_conditions", "prior_appts", "prior_noshows", "prior_noshow_rate"]
df[new_cols].head(10)

,wait_days,wait_group,same_day,age_group,appointment_weekday,scheduled_hour,num_conditions,prior_appts,prior_noshows,prior_noshow_rate
0,0,Same day,1,Senior (60-74),Friday,18,1,0,0,NaN
1,0,Same day,1,Middle age (45-59),Friday,16,0,0,0,NaN
2,0,Same day,1,Senior (60-74),Friday,16,0,0,0,NaN
3,0,Same day,1,Child (3-12),Friday,17,0,0,0,NaN
4,0,Same day,1,Middle age (45-59),Friday,16,2,0,0,NaN
5,2,1-3 days,0,Elderly (75+),Friday,8,1,0,0,NaN
6,2,1-3 days,0,Young adult (18-29),Friday,15,0,0,0,NaN
7,2,1-3 days,0,Adult (30-44),Friday,15,0,0,0,NaN
8,0,Same day,1,Young adult (18-29),Friday,8,0,0,0,NaN
9,2,1-3 days,0,Young adult (18-29),Friday,12,0,0,0,NaN


**Feature notes**
- `wait_days` = appointment date - booking date (usually the strongest predictor)
- `prior_appts` / `prior_noshows` = the patient's earlier history **using only appointments that had already taken place on the booking day**. This avoids *data leakage*: we never use information the hospital would not have had at booking time.
- `num_conditions` = hypertension + diabetes + alcoholism + handicap flag

## 5 · Validate

In [8]:
assert df["age"].min() >= 0
assert df["wait_days"].min() >= 0
assert df["missed"].isin([0, 1]).all()
assert df["appointment_id"].is_unique
assert (df["prior_noshows"] <= df["prior_appts"]).all()
print("All validation checks passed")
print(f"Final rows: {len(df):,} | columns: {df.shape[1]} | overall no-show rate: {df['missed'].mean():.1%}")

All validation checks passed
Final rows: 110,521 | columns: 32 | overall no-show rate: 20.2%


In [9]:
df.describe(include='all').T.head(40)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
patient_id,110521.0,NaN,NaN,NaN,147490629021863.6875,39217.0,4172457111246.0,31731846175313.0,94389629939367.0,999981631772427.0,256085972718815.59375
appointment_id,110521.0,NaN,NaN,NaN,5675303.982972,5030230.0,5640284.0,5680573.0,5725524.0,5790484.0,71296.911799
gender,110521,2,F,71836,NaN,NaN,NaN,NaN,NaN,NaN,NaN
scheduled_dt,110521,NaN,NaN,NaN,2016-05-09 07:48:50.890228,2015-11-10 07:13:56,2016-04-29 10:26:51,2016-05-10 12:13:17,2016-05-20 11:18:39,2016-06-08 20:07:23,NaN
appointment_date,110521,NaN,NaN,NaN,2016-05-19 00:58:20.684937,2016-04-29 00:00:00,2016-05-09 00:00:00,2016-05-18 00:00:00,2016-05-31 00:00:00,2016-06-08 00:00:00,NaN
age,110521.0,NaN,NaN,NaN,37.089386,0.0,18.0,37.0,55.0,115.0,23.109885
neighbourhood,110521,81,JARDIM CAMBURI,7717,NaN,NaN,NaN,NaN,NaN,NaN,NaN
scholarship,110521.0,NaN,NaN,NaN,0.098271,0.0,0.0,0.0,0.0,1.0,0.297682
hypertension,110521.0,NaN,NaN,NaN,0.197257,0.0,0.0,0.0,0.0,1.0,0.397929
diabetes,110521.0,NaN,NaN,NaN,0.071869,0.0,0.0,0.0,0.0,1.0,0.258272


## 6 · Save

In [10]:
df.to_csv(OUT, index=False)
print("Saved ->", OUT)

# Compact summary table (handy for Power BI / Excel users)
summary = (df.groupby("neighbourhood")
             .agg(appointments=("missed", "size"), no_shows=("missed", "sum"), no_show_rate=("missed", "mean"),
                  avg_wait_days=("wait_days", "mean"))
             .round(3).sort_values("appointments", ascending=False))
summary.to_csv("../data/processed/neighbourhood_summary.csv")
summary.head(10)

Saved -> ../data/processed/appointments_clean.csv


,appointments,no_shows,no_show_rate,avg_wait_days
neighbourhood,,,,
JARDIM CAMBURI,7717,1465,0.190,18.761
MARIA ORTIZ,5805,1219,0.210,13.367
RESISTÊNCIA,4430,905,0.204,12.749
JARDIM DA PENHA,3877,631,0.163,10.535
ITARARÉ,3514,923,0.263,11.007
CENTRO,3334,703,0.211,11.008
SANTA MARTHA,3131,496,0.158,8.466
TABUAZEIRO,3131,572,0.183,9.103
JESUS DE NAZARETH,2853,696,0.244,11.796
